# Create Rita Allen Awards (FELLOWSHIP PATTERN, method-5 static HTML)

Rita Allen Foundation Scholars from the foundation's own per-year archive at ritaallen.org/scholar-year/{YYYY}/. Includes the main Rita Allen Foundation Scholars program plus the "Award in Pain Recipient" sub-program embedded within the same archive.

**Prerequisites:**
- Run `scripts/local/rita_allen_to_s3.py` first.

**Data source:** https://ritaallen.org/wp-sitemap-taxonomies-scholar-year-1.xml (43 year-archive URLs 1976-2025) + per-year archive pages at https://ritaallen.org/scholar-year/{YYYY}/
**S3 location:** `s3a://openalex-ingest/awards/rita_allen/rita_allen_scholars.parquet`

**Awarding body:**
- funder_id: 4320306590
- display_name: "Rita Allen Foundation"
- country: US
- ROR: none
- DOI: 10.13039/100001447

**Coverage (from 2026-05-22 local `--skip-upload` run):**
- 222 scholar rows across 1976-2025 (43 year pages)
- 100% scholar_name / award_year / institution coverage
- 77.5% bio coverage (older cohorts pre-~2000 lack the modal bio paragraph)
- 222/222 unique `funder_award_id` (`rita-allen-{year}-{slug}`)
- 18 rows tagged as "Award in Pain Recipient" sub-program; 204 default to "Rita Allen Foundation Scholar"

**Amount/currency:** NULL with §6.7 waiver. The foundation publishes program-level amount ranges in narrative form on /programs pages (Scholars: ~$110k/yr × 5 yrs; Pain Scholars: ~$50k/yr × 3 yrs) but does NOT publish a verified per-scholar amount in any structured form on the scholar archive. Precedent: HHMI (priority 44, NULL), CIFAR (priority 79, NULL), Damon Runyon (priority 73, NULL).

**Provenance:** `rita_allen_scholars` (direct from the foundation's own site; not an aggregator).

**Priority:** 107 (UCOP at 106 is the next-lower head as of this PR; Vilcek at 105 is current registry head on main).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rita_allen_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rita_allen/rita_allen_scholars.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.rita_allen_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.rita_allen_raw;


## Step 1.5: Inspect Raw Data, Money Scan, and Native Key

Rita Allen Scholars are non-monetary in our schema (amount NULL by waiver per Fields Medal / HHMI / CIFAR / Damon Runyon precedent — the foundation does not publish per-scholar amounts on the archive). The money-field discovery scan is run for completeness; we expect zero matches because the parquet was built without monetary columns.

In [ ]:
%sql
-- Runbook §1.5 money-field discovery scan (expected: no matches; amount NULL by waiver)
SELECT column_name FROM (DESCRIBE openalex.awards.rita_allen_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
-- Runbook §1.5 currency-field discovery scan (expected: no matches)
SELECT column_name FROM (DESCRIBE openalex.awards.rita_allen_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
-- Sample 5 raw rows
SELECT
    scholar_name, award_year, slug, institution, program, lab_url,
    SUBSTR(bio, 1, 200) AS bio_preview
FROM openalex.awards.rita_allen_raw
LIMIT 5;


In [ ]:
%sql
-- Native key uniqueness — funder_award_id (rita-allen-{year}-{slug}) must be unique
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.rita_allen_raw;
-- collisions MUST be 0. If not, the parquet write would have failed; investigate.


In [ ]:
%sql
-- Year distribution sanity (expected 1976-2025, ~5 scholars/year on average)
SELECT award_year, COUNT(*) as scholars
FROM openalex.awards.rita_allen_raw
GROUP BY award_year
ORDER BY award_year DESC
LIMIT 30;


In [ ]:
%sql
-- Program distribution. Expected: ~204 NULL (default Scholar) + ~18 "Award in Pain Recipient".
SELECT
    COALESCE(program, '(default Rita Allen Foundation Scholar)') AS program,
    COUNT(*) AS rows
FROM openalex.awards.rita_allen_raw
GROUP BY program
ORDER BY rows DESC;


In [ ]:
%sql
-- Bio coverage by decade (older cohorts pre-2000 often lack the modal bio)
SELECT
    CONCAT(CAST(FLOOR(CAST(award_year AS INT) / 10) * 10 AS INT), 's') AS decade,
    COUNT(*) AS total,
    COUNT(bio) AS with_bio,
    ROUND(COUNT(bio) * 100.0 / COUNT(*), 1) AS pct_bio
FROM openalex.awards.rita_allen_raw
GROUP BY FLOOR(CAST(award_year AS INT) / 10)
ORDER BY decade;


## Step 1.6: Fail-fast — Verify the Rita Allen Funder Row Exists

The Step 2 transform CROSS JOINs against `openalex.common.funder` to populate the `funder` struct. If F4320306590 is absent, the join silently emits zero rows. **Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320306590;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rita_allen_awards
USING delta
AS
WITH
rita_allen_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306590
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.rita_allen_raw
    WHERE scholar_name IS NOT NULL
      AND TRIM(scholar_name) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        -- display_name = scholar name (the fellowship-pattern convention: the
        -- recipient themselves IS the award identifier from the user's perspective)
        g.scholar_name as display_name,

        -- description = bio when available; NULL otherwise
        g.bio as description,

        f.funder_id,
        g.funder_award_id,

        -- amount/currency NULL by §6.7 waiver (foundation doesn't publish
        -- per-scholar amounts on the archive). Per HHMI #44 / CIFAR #79 /
        -- Damon Runyon #73 precedent.
        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'fellowship' as funding_type,

        -- funder_scheme — the sub-program tag when detected, else the default
        COALESCE(
            CONCAT('Rita Allen Foundation ', NULLIF(TRIM(g.program), '')),
            'Rita Allen Foundation Scholar'
        ) as funder_scheme,

        'rita_allen_scholars' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,  -- 5-year program but end date not published per-scholar
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- lead_investigator = the scholar themselves (fellowship pattern,
        -- HHMI/CIFAR/Damon Runyon precedent)
        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.institution), '') as name,
                CAST(NULL AS STRING) as country,  -- US-based mostly but not exposed structured
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- landing_page_url = the per-year archive (the closest stable foundation page per scholar)
        g.scholar_year_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN rita_allen_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 107

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data (runbook §2.2.6).
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rita_allen_scholars' AND priority = 107;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    107 as priority  -- Rita Allen priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.rita_allen_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.rita_allen_awards;
-- Expected: 222 (matches the upload-script output).


In [ ]:
%sql
DESCRIBE openalex.awards.rita_allen_awards;


In [ ]:
%sql
-- §6.3 Data completeness. Expected: pct_title=100%, pct_amount=0% (waivered),
-- pct_description~77.5%, pct_pi=100%.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) as pct_pi
FROM openalex.awards.rita_allen_awards;


In [ ]:
%sql
-- §6.4 Sample inspection
SELECT
    funder_award_id, display_name, start_year, funder_scheme,
    lead_investigator.given_name AS pi_given,
    lead_investigator.family_name AS pi_family,
    lead_investigator.affiliation.name AS pi_affiliation,
    SUBSTR(description, 1, 120) AS desc_preview
FROM openalex.awards.rita_allen_awards
LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.rita_allen_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.rita_allen_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 50;


In [ ]:
%sql
-- §6.7 Amount/currency check — amount intentionally NULL by §6.7 waiver.
-- distinct_currencies should be 0; pct_amount should be 0.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.rita_allen_awards;


In [ ]:
%sql
-- Scheme distribution — verify program-tag detection (18 Pain Recipients + 204 default).
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.rita_allen_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;


In [ ]:
%sql
-- funder_award_id uniqueness across the awards table
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.rita_allen_awards;
